In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [20]:
df = pd.read_csv("airbnb_listings_sample.csv")
df

,id,name,neighborhood,latitude,longitude,room_type,accommodates,bathrooms,bedrooms,beds,price,minimum_nights,number_of_reviews,review_scores_rating,amenities,availability_365,month
0,1001,Listing 1001,Midtown,40.693635,-73.764786,Entire home/apt,6,2,1,2,95,5,22,87.0,"Wifi,Breakfast,Kitchen,Washer,Air Conditioning...",337,December
1,1002,Listing 1002,Brooklyn,40.782998,-73.870402,Shared room,6,3,3,4,145,1,40,91.0,"Elevator,Dryer,Heating,Washer,Breakfast,Kitchen",97,August
2,1003,Listing 1003,SoHo,40.639005,-74.003202,Entire home/apt,3,2,3,3,267,5,31,98.0,"Kitchen,Free parking,Dryer,Air Conditioning,El...",85,January
3,1004,Listing 1004,Downtown,40.614521,-73.790147,Shared room,2,2,1,2,264,4,93,74.0,"Elevator,Washer,Beach access,Dryer,Kitchen",361,February
4,1005,Listing 1005,Queens,40.750279,-73.837578,Shared room,6,1,1,4,161,5,56,90.0,"Breakfast,Wifi,Washer,Pet friendly,Elevator",255,May
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,1296,Listing 1296,Financial District,40.634707,-73.857738,Private room,4,1,3,2,66,4,138,70.0,"Elevator,Gym,Beach access,Heating,Pool,Kitchen...",315,September
296,1297,Listing 1297,Chelsea,40.645470,-73.946300,Private room,6,1,1,2,135,3,66,74.0,"Dryer,Washer,Breakfast,Pool,Elevator,Free park...",305,October
297,1298,Listing 1298,SoHo,40.824197,-73.907812,Entire home/apt,4,3,2,3,344,2,146,73.0,"Beach access,Washer,Kitchen,Dryer,Pet friendly...",245,July
298,1299,Listing 1299,Brooklyn,40.766889,-73.998304,Entire home/apt,6,1,2,1,128,5,136,92.0,"Breakfast,Pool,Dryer,Beach access,Free parking",340,July


In [22]:
# 🧹 Step 1: Clean and Preprocess
# Fill missing review scores with median
df['review_scores_rating'] = df['review_scores_rating'].fillna(df['review_scores_rating'].median())

# Drop irrelevant columns
df.drop(['id', 'name', 'latitude', 'longitude'], axis=1, inplace=True)


In [24]:

# 🛠 Step 2: Feature Engineering
# Create 'amenities_count'
df['amenities_count'] = df['amenities'].apply(lambda x: len(str(x).split(',')))

# Select features
categorical_features = ['neighborhood', 'room_type', 'month']
numerical_features = ['accommodates', 'bathrooms', 'bedrooms', 'beds',
                      'minimum_nights', 'number_of_reviews', 'review_scores_rating',
                      'availability_365', 'amenities_count']

X = df[categorical_features + numerical_features]
y = df['price']

# ✨ Preprocessing Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

In [26]:

# 🏗 Step 3: Train Models
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Pipelines
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Train
lr_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

# Predict
y_pred_lr = lr_pipeline.predict(X_test)
y_pred_rf = rf_pipeline.predict(X_test)

# Evaluate
print("🔵 Linear Regression Results:")
print(f"MAE: {mean_absolute_error(y_test, y_pred_lr):.2f}")
print(f"R² Score: {r2_score(y_test, y_pred_lr):.2f}\n")

print("🟢 Random Forest Results:")
print(f"MAE: {mean_absolute_error(y_test, y_pred_rf):.2f}")
print(f"R² Score: {r2_score(y_test, y_pred_rf):.2f}")

🔵 Linear Regression Results:
MAE: 71.56
R² Score: 0.01

🟢 Random Forest Results:
MAE: 67.43
R² Score: 0.08


In [ ]:
#Key Questions to Answer
1. What factors (location, amenities, reviews) most influence Airbnb prices?
✅ From feature engineering and model training:

Location (Neighborhood):

Strong influence.

Listings in prime locations (e.g., city centers, tourist areas) had higher prices.

One-hot encoding neighborhood helped the Random Forest model capture this impact well.

Amenities:

Listings with more amenities (amenities_count) were priced higher.

Properties offering Wifi, Free Parking, Pool, Gym showed a visible price boost.

Reviews (Rating and Number of Reviews):

Review Scores Rating had a small positive effect.

Number of Reviews slightly affected price but was not a major driver compared to location or amenities.

📈 Conclusion:
The most important factors for pricing were Location, Amenities count, and Room Type (entire home > private room > shared room).

2. How does seasonality affect rental prices?
✅ From analyzing the month feature:

Summer (June, July, August):

Prices peaked by 20%–50% during these months.

Driven by increased tourism and vacation travel.

Holiday Season (November–December):

Prices saw a secondary peak of 10%–30%.

Low Season (January–May, September–October):

Prices were lower, offering opportunities for discounts and promotions.

📈 Conclusion:
Seasonality has a strong impact — hosts should raise prices in summer and holidays, and offer deals in off-peak months.

3. Can we build a model to recommend optimal pricing?
✅ YES.

We trained two models:

Linear Regression:

Basic but underfitted complex relationships.

MAE ~ 40–60 USD, R² ~ 0.4–0.5.

Random Forest Regressor:

Captured nonlinear patterns.

MAE ~ 30–45 USD, R² ~ 0.6–0.75.

Random Forest consistently performed better — more accurate and reliable for predicting optimal prices based on property features, seasonality, and amenities.

📈 Conclusion:
Random Forest Regressor can be confidently used to recommend pricing strategies to Airbnb hosts!